In [ ]:
import os
import sys
from pathlib import Path

# Get the notebook's directory
NOTEBOOK_DIR = Path(os.getcwd())
BACKEND_DIR = NOTEBOOK_DIR.parent.parent / "Skin_Lesion_Classification_backend"
ML_DIR = BACKEND_DIR / "ml"

# Add backend ml/ to sys.path so imports like `from src.data.dataset` work
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Backend directory:  {BACKEND_DIR}")
print(f"ML directory:      {ML_DIR}")
print(f"Python:            {sys.executable}")

# Verify backend exists
if not BACKEND_DIR.exists():
    print(f"\u274c ERROR: Backend not found at {BACKEND_DIR}")
    print("Make sure the backend repo is cloned at the same level as the frontend repo.")
else:
    print("\u2705 Backend directory found.")

# Paper Results Table — Compile All RQ Results
## XAI Skin Lesion Classification — Final Results

**Purpose**: Once all RQ notebooks have been run, compile results into
paper-ready tables with LaTeX export.

**Run order**: Run this AFTER all other RQ notebooks have completed.

---

## CELL 1: Load All Results

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

NOTEBOOK_DIR = Path(os.getcwd())
OUTPUTS_DIR = NOTEBOOK_DIR.parent / "outputs" / "metrics"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

METRICS_DIR = NOTEBOOK_DIR.parent / "outputs" / "metrics"

# Load results with graceful handling of missing files
rq1_path = METRICS_DIR / "RQ1_cam_comparison.csv"
rq2_path = METRICS_DIR / "RQ2_faithfulness.csv"
rq3_path = METRICS_DIR / "RQ3_backbone_comparison.csv"
rq4_path = METRICS_DIR / "RQ4_agreement_results.csv"

rq1 = pd.read_csv(rq1_path) if rq1_path.exists() else None
rq2 = pd.read_csv(rq2_path) if rq2_path.exists() else None
rq3 = pd.read_csv(rq3_path) if rq3_path.exists() else None
rq4 = pd.read_csv(rq4_path) if rq4_path.exists() else None

print("=== Data Loaded ===")
available = []
for name, df in [("RQ1 (CAM comparison)", rq1), ("RQ2 (Faithfulness)", rq2), 
                  ("RQ3 (Backbone XAI)", rq3), ("RQ4 (Agreement)", rq4)]:
    status = f"{len(df)} rows" if df is not None else "MISSING"
    print(f"  {name}: {status}")
    if df is not None:
        available.append(name.split()[0])

if not available:
    print("\n❌ No results found. Run RQ1-RQ4 notebooks first to generate results.")

---

## CELL 2: Table 1 — RQ1 CAM Method Comparison

In [ ]:
# ─── Self-contained: check data availability ───
if rq1 is None:
    print("❌ RQ1 data not available. Run RQ1 notebook first.")
else:
    table1 = rq1.groupby('method')[['fap_05', 'entropy']].agg(['mean', 'std'])
    table1.columns = ['FAP Mean', 'FAP Std', 'Entropy Mean', 'Entropy Std']
    print("=== Table 1: CAM Method Comparison (RQ1) ===")
    print("(Focus Area % = lower is better; Entropy = lower is better)")
    print()
    print(table1.round(4))

    # LaTeX export
    print("\n=== LaTeX ===")
    print(table1.to_latex(float_format="%.4f"))

---

## CELL 3: Table 2 — RQ2 Faithfulness Metrics

In [ ]:
# ─── Self-contained: check data availability ───
if rq2 is None:
    print("❌ RQ2 data not available. Run RQ2 notebook first.")
else:
    table2 = rq2.groupby('method')[['insertion_auc', 'deletion_auc', 'faithfulness']].agg(['mean', 'std'])
    table2.columns = ['Insertion AUC Mean', 'Insertion AUC Std', 'Deletion AUC Mean', 'Deletion AUC Std', 'Faithfulness Mean', 'Faithfulness Std']
    table2_simple = rq2.groupby('method')[['insertion_auc', 'deletion_auc', 'faithfulness']].mean()
    print("=== Table 2: Faithfulness Metrics (RQ2) ===")
    print("(Insertion AUC = higher is better; Deletion AUC = lower is better; Faithfulness = higher is better)")
    print()
    print(table2_simple.round(4))

    print("\n=== LaTeX ===")
    print(table2.to_latex(float_format="%.4f"))

---

## CELL 4: Table 3 — RQ3 Backbone × CAM Method

In [ ]:
# ─── Self-contained: check data availability ───
if rq3 is None:
    print("❌ RQ3 data not available. Run RQ3 notebook first.")
else:
    table3 = rq3[rq3['correct']==1].groupby(['model', 'cam_method'])['fap_05'].mean().unstack()
    print("=== Table 3: Backbone × CAM Method — Focus Area % (RQ3) ===")
    print("(Lower = more focused attention)")
    print()
    print(table3.round(4))

    print("\n=== LaTeX ===")
    print(table3.to_latex(float_format="%.4f"))

---

## CELL 5: Table 4 — RQ4 Agreement vs Correctness

In [ ]:
# ─── Self-contained: check data availability ───
if rq4 is None:
    print("❌ RQ4 data not available. Run RQ4 notebook first.")
else:
    correct_agree   = rq4[rq4['correct']==1]['avg_jaccard']
    incorrect_agree = rq4[rq4['correct']==0]['avg_jaccard']

    print("=== Table 4: Inter-method Agreement (RQ4) ===")
    print()
    print(f"Correct predictions:   Jaccard = {correct_agree.mean():.4f} ± {correct_agree.std():.4f} (n={len(correct_agree)})")
    print(f"Incorrect predictions: Jaccard = {incorrect_agree.mean():.4f} ± {incorrect_agree.std():.4f} (n={len(incorrect_agree)})")
    print(f"Difference:            {correct_agree.mean() - incorrect_agree.mean():.4f}")

    from scipy.stats import mannwhitneyu
    stat, p = mannwhitneyu(correct_agree, incorrect_agree, alternative='greater')
    print(f"Mann-Whitney U p-value: {p:.4f}")

    print("\n=== LaTeX ===")
    print(f"Correct & {correct_agree.mean():.4f} \\pm {correct_agree.std():.4f} \\")
    print(f"Incorrect & {incorrect_agree.mean():.4f} \\pm {incorrect_agree.std():.4f} \\")
    print(f"p-value & {p:.4f} \\")

---

## CELL 6: Full Results Summary

In [ ]:
print("========================================")
print("       RESEARCH RESULTS SUMMARY")
print("========================================")
print()

if rq1 is not None:
    print("RQ1: Which CAM variant best localizes?")
    best_fap = rq1.groupby('method')['fap_05'].mean().idxmin()
    print(f"  Best method (FAP): {best_fap} ({rq1.groupby('method')['fap_05'].mean()[best_fap]:.4f})")
    best_entropy = rq1.groupby('method')['entropy'].mean().idxmin()
    print(f"  Best method (entropy): {best_entropy} ({rq1.groupby('method')['entropy'].mean()[best_entropy]:.4f})")
else:
    print("RQ1: No data available")

if rq2 is not None:
    print()
    print("RQ2: Faithfulness metrics")
    best_faith = rq2.groupby('method')['faithfulness'].mean().idxmax()
    print(f"  Most faithful: {best_faith} ({rq2.groupby('method')['faithfulness'].mean()[best_faith]:.4f})")
else:
    print("RQ2: No data available")

if rq3 is not None:
    print()
    print("RQ3: Backbone effect on XAI quality")
    backbone_aucs = rq3[rq3['correct']==1].groupby('model')['fap_05'].mean()
    print(f"  Most focused backbone: {backbone_aucs.idxmin()} ({backbone_aucs.min():.4f})")
else:
    print("RQ3: No data available")

if rq4 is not None:
    print()
    print("RQ4: Agreement predicts correctness")
    correct_agree   = rq4[rq4['correct']==1]['avg_jaccard']
    incorrect_agree = rq4[rq4['correct']==0]['avg_jaccard']
    print(f"  Correct: {correct_agree.mean():.4f} | Incorrect: {incorrect_agree.mean():.4f}")
    print(f"  Direction supports hypothesis: {correct_agree.mean() > incorrect_agree.mean()}")
else:
    print("RQ4: No data available")

print()
print("========================================")